<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

In [1]:
!pip install gradio google-generativeai pandas matplotlib -q

In [2]:
import gradio as gr
import pandas as pd
import gspread
import datetime
import matplotlib.pyplot as plt
from google.colab import auth
from google.auth import default

In [3]:
# 1. 安裝中文字體
!sudo apt-get install -y fonts-noto-cjk

# 2. 清除 matplotlib 字體快取
!rm -rf ~/.cache/matplotlib

# 3. 強制讓 matplotlib 讀取新安裝的字體檔案
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 搜尋系統中的 Noto Sans CJK 字體檔案並加入 FontManager
font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/opentype/noto'])
for font_file in font_files:
    fm.fontManager.addfont(font_file)

# 4. 配置使用中文字體
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP']
plt.rcParams['axes.unicode_minus'] = False

print("字體重新載入完成，請執行下方的測試區塊。")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  fonts-noto-cjk-extra
The following NEW packages will be installed:
  fonts-noto-cjk
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 61.2 MB of archives.
After this operation, 93.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-cjk all 1:20220127+repack1-1 [61.2 MB]
Fetched 61.2 MB in 3s (17.6 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fon

In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_URL = 'https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing'
gsheets = gc.open_by_url(SHEET_URL)
worksheet = gsheets.worksheet('工作表1')

In [9]:
import os
import gradio as gr
import pandas as pd
import gspread
import datetime
import matplotlib.pyplot as plt
import google.generativeai as genai
from google.colab import auth
from google.auth import default
from google.colab import userdata

# --- 3. 功能函式定義 ---

def get_raw_data():
    """讀取試算表原始資料並回傳 DataFrame"""
    data = worksheet.get_all_values()
    col_names = ["日期", "時間", "分類", "品項", "金額", "地點", "支付方式", "付款人/分攤成員", "實付金額"]

    if len(data) <= 1:
        return pd.DataFrame(columns=col_names)

    df = pd.DataFrame(data[1:], columns=data[0])
    if len(df.columns) == 9:
        df.columns = col_names

    return df

def get_filtered_data(start_date=None, end_date=None):
    """取得經過日期過濾的 DataFrame"""
    df = get_raw_data()
    if df.empty:
        return df

    df['日期格式'] = pd.to_datetime(df['日期'], errors='coerce')

    if start_date:
        try:
            df = df[df['日期格式'] >= pd.to_datetime(start_date)]
        except:
            pass

    if end_date:
        try:
            df = df[df['日期格式'] <= pd.to_datetime(end_date)]
        except:
            pass
    return df

def get_pie_chart(start_date=None, end_date=None):
    """生成圓餅圖 (依實付金額)"""
    df = get_filtered_data(start_date, end_date)
    fig, ax = plt.subplots(figsize=(6, 6))

    if df.empty:
        ax.text(0.5, 0.5, "尚無資料", ha='center', va='center', fontsize=12)
        ax.axis('off')
        return fig

    df['實付金額'] = pd.to_numeric(df['實付金額'], errors='coerce').fillna(0)
    category_sums = df.groupby('分類')['實付金額'].sum()

    if category_sums.sum() == 0:
        ax.text(0.5, 0.5, "此日期區間無支出紀錄", ha='center', va='center', fontsize=12)
        ax.axis('off')
        ax.set_title('支出分類佔比 (依實付金額計算)')
        return fig

    category_sums.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=140)
    ax.set_ylabel('')

    title = '支出分類佔比 (依實付金額計算)'
    if start_date or end_date:
        title += f'\n({start_date or "最早"} ~ {end_date or "最新"})'
    ax.set_title(title)

    return fig

def submit_to_sheet(date_str, time_str, category, item, amount, location, payment_method, split_count, payer_info, filter_start, filter_end):
    """寫入資料並返回狀態"""
    try:
        datetime.datetime.strptime(date_str, '%Y-%m-%d')
        split_count = int(split_count) if split_count and split_count > 0 else 1
        amount = float(amount) if amount else 0.0
        final_amount = round(amount / split_count)

        if split_count > 1:
            member_note = f"{payer_info} ({split_count}人AA)" if payer_info else f"{split_count}人AA"
        else:
            member_note = payer_info

        new_row = [[date_str, time_str, category, item, str(amount), location, payment_method, member_note, str(final_amount)]]
        worksheet.append_rows(values=new_row, value_input_option='USER_ENTERED')

        new_plot = get_pie_chart(filter_start, filter_end)
        new_df = get_raw_data()

        if split_count > 1:
            status_msg = f"成功寫入！{item} (總價 ${amount}，實付 ${final_amount})"
        else:
            status_msg = f"成功寫入！{item} (實付 ${final_amount})"

        return status_msg, new_plot, new_df
    except Exception as e:
        return f"發生錯誤：{str(e)}", None, None

def generate_ai_advice(start_date, end_date):
    """呼叫 Gemini API 生成理財建議"""
    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        return "請先在環境變數中設定 GEMINI_API_KEY！"

    df = get_filtered_data(start_date, end_date)
    if df.empty:
        return "這段期間沒有任何消費紀錄，AI 無法產生建議。"

    df['實付金額'] = pd.to_numeric(df['實付金額'], errors='coerce').fillna(0)
    category_sums = df.groupby('分類')['實付金額'].sum()
    total_spent = category_sums.sum()

    if total_spent == 0:
        return "這段期間實付總額為 0，無需分析。"

    # 彙整消費數據作為 Prompt
    summary_text = f"總支出：{total_spent} 元\n"
    for cat, amt in category_sums.items():
        if amt > 0:
            summary_text += f"- {cat}: {amt} 元\n"

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.5-flash')
        prompt = f"這是我從 {start_date} 到 {end_date} 的日常花費統計：\n{summary_text}\n請你扮演專業的理財顧問，用繁體中文分析我的花費習慣，並給予我 2 到 3 個具體、實用的省錢或理財建議。語氣要專業且鼓勵人。"

        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"AI 分析時發生錯誤，請檢查 API Key 或網路連線。\n詳細錯誤：{str(e)}"

# --- 4. Gradio 介面配置 ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 日常支出速算與 AI 理財分析")

    # 取得今天與當月1號作為預設時間
    today = datetime.date.today()
    first_day = today.replace(day=1)

    with gr.Tab("支出紀錄"):
        date_in = gr.Textbox(label="日期 (YYYY-MM-DD)", value=today.isoformat())
        time_in = gr.Textbox(label="時間", value=datetime.datetime.now().strftime("%H:%M"))
        cat_in = gr.Dropdown(choices=["食物", "飲料", "交通", "書", "其他"], label="支出類別", value="食物")
        item_in = gr.Textbox(label="品項")

        with gr.Row():
            amt_in = gr.Number(label="總金額")
            split_count_in = gr.Number(label="拆帳人數", value=1, minimum=1, precision=0)

        location_in = gr.Textbox(label="地點")
        payment_in = gr.Dropdown(choices=["現金", "信用卡", "電子支付", "轉帳"], label="支付方式", value="現金")
        payer_in = gr.Textbox(label="付款人/分攤成員", placeholder="例如：先代墊的小明，或是與誰平分")

        btn = gr.Button("記錄支出", variant="primary")
        status = gr.Textbox(label="狀態")

    with gr.Tab("支出分析圖"):
        with gr.Row():
            filter_start = gr.Textbox(label="開始日期 (YYYY-MM-DD)", value=first_day.isoformat())
            filter_end = gr.Textbox(label="結束日期 (YYYY-MM-DD)", value=today.isoformat())

        refresh_chart_btn = gr.Button("依日期區間重新繪製圖表", variant="secondary")
        plot_output = gr.Plot(label="支出分類佔比")

    with gr.Tab("AI 理財建議"):
        gr.Markdown("系統將自動讀取環境變數中的 GEMINI_API_KEY 進行分析。")

        with gr.Row():
            ai_start_date = gr.Textbox(label="分析開始日期", value=first_day.isoformat())
            ai_end_date = gr.Textbox(label="分析結束日期", value=today.isoformat())

        ai_btn = gr.Button("產生專屬理財報告", variant="primary")
        ai_output = gr.Markdown(label="AI 分析結果", value="分析報告將顯示於此...")

    with gr.Tab("檢視原始資料"):
        data_table = gr.DataFrame(label="Google Sheet 原始資料")
        refresh_table_btn = gr.Button("重新整理資料表")

    # --- 事件綁定 ---
    btn.click(
        fn=submit_to_sheet,
        inputs=[date_in, time_in, cat_in, item_in, amt_in, location_in, payment_in, split_count_in, payer_in, filter_start, filter_end],
        outputs=[status, plot_output, data_table]
    )
    refresh_chart_btn.click(fn=get_pie_chart, inputs=[filter_start, filter_end], outputs=plot_output)
    ai_btn.click(fn=generate_ai_advice, inputs=[ai_start_date, ai_end_date], outputs=ai_output)
    refresh_table_btn.click(fn=get_raw_data, outputs=data_table)

    # 啟動預載
    demo.load(fn=get_pie_chart, inputs=[filter_start, filter_end], outputs=plot_output)
    demo.load(fn=get_raw_data, outputs=data_table)

demo.launch(debug=True, share=True)

/tmp/ipykernel_1739/1951887507.py:140: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3409fdd5c93c9feda.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b3409fdd5c93c9feda.gradio.live
